# **Chemprop Hyperparameter Optimization**

- This notebook performs hyperparameter optimization for Chemprop models used to predict molecular activity values (pIC50) for a selected biological target.

- Chemprop is installed from GitHub when running in a Google Colab environment.

- Hyperparameter optimization is executed using Chemprop’s built-in hpopt utility with Ray Tune, which performs automated search across a predefined set of model parameters. A basic search space is explored using multiple trials, where each trial trains a Chemprop model with a different hyperparameter configuration.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
#!pip install chemprop

In [ ]:
# Install chemprop from GitHub if running in Google Colab
import os

if os.getenv("COLAB_RELEASE_TAG"):
    try:
        import chemprop
    except ImportError:
        !git clone https://github.com/chemprop/chemprop.git
        %cd chemprop
        !pip install .
        %cd examples

Cloning into 'chemprop'...
remote: Enumerating objects: 25457, done.
remote: Counting objects: 100% (205/205), done.
remote: Compressing objects: 100% (164/164), done.
remote: Total 25457 (delta 117), reused 41 (delta 40), pack-reused 25252 (from 3)
Receiving objects: 100% (25457/25457), 876.24 MiB | 36.23 MiB/s, done.
Resolving deltas: 100% (18241/18241), done.
Updating files: 100% (337/337), done.
/content/chemprop
Processing /content/chemprop
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.5/828.5 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.1/36.1 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

In [ ]:
!pip install -U ray[tune]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.1/70.1 MB 13.1 MB/s eta 0:00:00


In [ ]:
!pip install -e .[hpopt]

Obtaining file:///content/chemprop/examples
ERROR: file:///content/chemprop/examples does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


In [ ]:
# pick a target
targets = ["5-HT6","ache", "bace1", "buche","esr1","gsk-3beta", "mao-b"]
target = targets[4]

# Build paths with f-strings (note the leading f)
DATA_PATH = f"/content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/chemprop/input_data/{target}_train.csv"
SAVE_DIR  = f"/content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/chemprop/hyperopt_models/{target}"

import os
os.makedirs(SAVE_DIR, exist_ok=True)
assert os.path.exists(DATA_PATH), f"Missing file: {DATA_PATH}"

print("Using:", DATA_PATH, "\nSaving to:", SAVE_DIR)


Using: /content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/chemprop/input_data/esr1_train.csv 
Saving to: /content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/chemprop/hyperopt_models/esr1


In [ ]:
# 10-trial basic search, 1 GPU per trial
!chemprop hpopt \
  --data-path "$DATA_PATH" \
  --task-type regression \
  --smiles-columns smiles \
  --target-columns pIC50 \
  --search-parameter-keywords basic \
  --raytune-num-samples 10 \
  --raytune-use-gpu \
  --raytune-num-gpus 1 \
  --accelerator gpu \
  --devices 1 \
  --hpopt-save-dir "$SAVE_DIR" \
  --logfile



Streaming output truncated to the last 5000 lines.
│ train_loop_config/ffn_num_layers          1 │
│ train_loop_config/message_hidden_dim   2300 │
╰─────────────────────────────────────────────╯
(RayTrainWorker pid=34441) Setting up process group for: env:// [rank=0, world_size=1]
(TorchTrainer pid=34324) Started distributed worker processes: 
(TorchTrainer pid=34324) - (node_id=abd3b54f9efb3162be3fd30e70a17e84bbced930f34b9e49e69f8863, ip=172.28.0.12, pid=34441) world_rank=0, local_rank=0, node_rank=0

Trial status: 5 TERMINATED | 1 RUNNING | 1 PENDING
Current time: 2025-09-29 09:17:17. Total running time: 24min 32s
Logical resource usage: 0/2 CPUs, 1.0/1 GPUs (0.0/1.0 accelerator_type:T4)
Current best trial: c9500ed6 with val_loss=0.3653995096683502 and params={'train_loop_config': {'ffn_hidden_dim': 2000, 'ffn_num_layers': 2, 'depth': 3, 'message_hidden_dim': 1200, 'dropout': np.float64(0.05)}}
╭─────────────────────────────────────────────────────────────────────────────────────────

In [ ]:
"""

!chemprop hpopt \
  --data-path "$DATA_PATH" \
  --task-type regression \
  --search-parameter-keywords depth ffn_num_layers message_hidden_dim \
  --hpopt-save-dir "$SAVE_DIR" """


'\n\n!chemprop hpopt   --data-path "$DATA_PATH"   --task-type regression   --search-parameter-keywords depth ffn_num_layers message_hidden_dim   --hpopt-save-dir "$SAVE_DIR" '

In [ ]:
"""!chemprop train --data-path tests/data/regression.csv \
    --task-type regression \
    --output-dir solubility_checkpoints """